# Checkpoint 5 — RFM Customer Behavioral Analysis
## AI-Powered E-commerce Customer Segmentation and Churn Analysis
### Brazilian E-Commerce Public Dataset by Olist

This notebook performs customer-level RFM (Recency, Frequency, Monetary) behavioral analysis for the **93,358 delivered customers** established in Checkpoints 2 and 3. It explores the empirical distributions of $R, F, M$, demonstrates why conventional independent quintile scoring encounters quantile collapse on Frequency, evaluates candidate scoring strategies, defines explicit rule-based behavioral segments, and constructs leakage-safe cutoff-based RFM features at 120-day and 180-day horizons.

## 1. Objective & Methodological Scope

### Research Questions
1. How are Recency, Frequency, and Monetary values distributed across the operational customer population?
2. What are the empirical consequences of the 97.00% single-purchase customer concentration on standard RFM quintile binning?
3. What candidate scoring schemes effectively differentiate customer engagement without arbitrary score inflation?
4. How can customers be segmented into reproducible, actionable behavioral cohorts using explicit domain rules?
5. How do RFM distributions look when evaluated strictly prior to candidate observation cutoffs ($T_{obs}^{120d}$ and $T_{obs}^{180d}$) to ensure leakage-safe future churn modeling?

> **Methodological Boundary**: This checkpoint evaluates descriptive customer behavior and rule-based segmentation. It does **not** perform K-Means clustering (evaluated in Checkpoint 6), train predictive models, or select the final churn window.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rfm_analysis import run_rfm_analysis_audit

# Execute or load the aggregate checkpoint
audit_data = run_rfm_analysis_audit(project_root=project_root)
meta = audit_data['population_metadata']
rfm_dists = audit_data['rfm_distributions']

print(f"Project root: {project_root}")
print(f"Checkpoint: {audit_data['checkpoint']}")
print(f"Audit UTC: {audit_data['audit_timestamp_utc']}")
print(f"Delivered orders analyzed: {meta['delivered_orders_analyzed']:,}")
print(f"Unique delivered customers: {meta['unique_customers_analyzed']:,}")
print(f"Reference timestamp: {meta['rfm_reference_timestamp']}")
print(f"Raw data immutability verified: {audit_data['raw_data_immutability_verified']}")


## 2. Customer-Level RFM Distribution Summary

Recency is measured as elapsed fractional days from the latest delivered purchase to the dataset reference timestamp (`2018-08-29 15:00:37`). Frequency is the count of delivered orders. Monetary value is the total observed customer spend (`item_plus_freight` from Checkpoint 3). All raw variables are preserved without clipping or winsorizing.

In [ ]:
r_stat = rfm_dists['recency_days']
f_stat = rfm_dists['frequency_orders']
m_stat = rfm_dists['monetary_item_plus_freight_brl']
m_item = rfm_dists['monetary_item_price_only_brl']

df_rfm_summary = pd.DataFrame([
    {
        'Metric': 'Recency (Days)',
        'Count': f"{r_stat['count']:,}",
        'Missing': r_stat['missing_count'],
        'Min': f"{r_stat['min']:.2f}",
        'p25': f"{r_stat['p25']:.2f}",
        'Median': f"{r_stat['median']:.2f}",
        'Mean': f"{r_stat['mean']:.2f}",
        'p75': f"{r_stat['p75']:.2f}",
        'p90': f"{r_stat['p90']:.2f}",
        'p95': f"{r_stat['p95']:.2f}",
        'Max': f"{r_stat['max']:.2f}",
        'Std': f"{r_stat['std']:.2f}",
        'Skewness': f"{r_stat['skewness']:.2f}"
    },
    {
        'Metric': 'Frequency (Orders)',
        'Count': f"{f_stat['count']:,}",
        'Missing': f_stat['missing_count'],
        'Min': f"{f_stat['min']:.0f}",
        'p25': f"{f_stat['p25']:.0f}",
        'Median': f"{f_stat['median']:.0f}",
        'Mean': f"{f_stat['mean']:.4f}",
        'p75': f"{f_stat['p75']:.0f}",
        'p90': f"{f_stat['p90']:.0f}",
        'p95': f"{f_stat['p95']:.0f}",
        'Max': f"{f_stat['max']:.0f}",
        'Std': f"{f_stat['std']:.2f}",
        'Skewness': f"{f_stat['skewness']:.2f}"
    },
    {
        'Metric': 'Monetary (Item+Freight BRL)',
        'Count': f"{m_stat['count']:,}",
        'Missing': m_stat['missing_count'],
        'Min': f"{m_stat['min']:.2f}",
        'p25': f"{m_stat['p25']:.2f}",
        'Median': f"{m_stat['median']:.2f}",
        'Mean': f"{m_stat['mean']:.2f}",
        'p75': f"{m_stat['p75']:.2f}",
        'p90': f"{m_stat['p90']:.2f}",
        'p95': f"{m_stat['p95']:.2f}",
        'Max': f"{m_stat['max']:.2f}",
        'Std': f"{m_stat['std']:.2f}",
        'Skewness': f"{m_stat['skewness']:.2f}"
    },
    {
        'Metric': 'Monetary (Item Price Only BRL)',
        'Count': f"{m_item['count']:,}",
        'Missing': m_item['missing_count'],
        'Min': f"{m_item['min']:.2f}",
        'p25': f"{m_item['p25']:.2f}",
        'Median': f"{m_item['median']:.2f}",
        'Mean': f"{m_item['mean']:.2f}",
        'p75': f"{m_item['p75']:.2f}",
        'p90': f"{m_item['p90']:.2f}",
        'p95': f"{m_item['p95']:.2f}",
        'Max': f"{m_item['max']:.2f}",
        'Std': f"{m_item['std']:.2f}",
        'Skewness': f"{m_item['skewness']:.2f}"
    }
])
df_rfm_summary


## 3. Frequency Concentration & Repeat Purchase Reality

Evaluation of customer order frequency tiers and concentration of delivered orders and observed monetary value between one-time and repeat buyers.

In [ ]:
fc = audit_data['frequency_concentration']
fb = fc['frequency_breakdown']

df_freq_tiers = pd.DataFrame([
    {'Frequency Tier': 'F = 1 (One-Time)', 'Customer Count': f"{fb['F_eq_1']['customer_count']:,}", 'Share of Customer Base': f"{fb['F_eq_1']['customer_percentage']:.2f}%"},
    {'Frequency Tier': 'F = 2', 'Customer Count': f"{fb['F_eq_2']['customer_count']:,}", 'Share of Customer Base': f"{fb['F_eq_2']['customer_percentage']:.2f}%"},
    {'Frequency Tier': 'F = 3', 'Customer Count': f"{fb['F_eq_3']['customer_count']:,}", 'Share of Customer Base': f"{fb['F_eq_3']['customer_percentage']:.2f}%"},
    {'Frequency Tier': 'F = 4', 'Customer Count': f"{fb['F_eq_4']['customer_count']:,}", 'Share of Customer Base': f"{fb['F_eq_4']['customer_percentage']:.2f}%"},
    {'Frequency Tier': 'F = 5', 'Customer Count': f"{fb['F_eq_5']['customer_count']:,}", 'Share of Customer Base': f"{fb['F_eq_5']['customer_percentage']:.4f}%"},
    {'Frequency Tier': 'F >= 5 (Tail)', 'Customer Count': f"{fb['F_ge_5']['customer_count']:,}", 'Share of Customer Base': f"{fb['F_ge_5']['customer_percentage']:.4f}%"}
])

print('=== REPEAT CUSTOMER SUMMARY ===')
rep = fc['repeat_customer_summary']
print(f"One-Time Customers (F=1): {rep['one_time_customers_count']:,} ({rep['one_time_customers_pct']:.2f}%)")
print(f"Repeat Customers (F>=2):   {rep['repeat_customers_count']:,} ({rep['repeat_customers_pct']:.2f}%)")
print(f"Max Frequency Observed:    {rep['maximum_frequency_observed']}")

vc = fc['value_concentration']
print('=== VALUE CONCENTRATION ===')
print(f"Delivered Orders: One-Time = {vc['orders_delivered']['one_time_orders_pct']:.2f}% | Repeat = {vc['orders_delivered']['repeat_orders_pct']:.2f}%")
print(f"Observed Monetary Value: One-Time = {vc['observed_customer_monetary_value']['one_time_value_pct']:.2f}% | Repeat = {vc['observed_customer_monetary_value']['repeat_value_pct']:.2f}%")

df_freq_tiers


## 4. Monetary Value Distribution & Log-Transformation Analysis

Raw observed monetary value exhibits severe positive skewness (skew = 9.21), driven by transactions up to 13,664.08 BRL. Applying $\log_{1p}(M)$ substantially reduces monetary-value skewness (from 9.21 to 0.53) and will be evaluated as a candidate transformation for distance-based clustering.


In [ ]:
log_m = rfm_dists['log1p_monetary']

df_trans_comp = pd.DataFrame([
    {
        'Variable': 'Raw Monetary Value (M)',
        'Min': f"{m_stat['min']:.2f}",
        'Median': f"{m_stat['median']:.2f}",
        'Mean': f"{m_stat['mean']:.2f}",
        'p95': f"{m_stat['p95']:.2f}",
        'Max': f"{m_stat['max']:.2f}",
        'Std': f"{m_stat['std']:.2f}",
        'Skewness': f"{m_stat['skewness']:.2f}"
    },
    {
        'Variable': 'Log1p Transformed Monetary (log1p_M)',
        'Min': f"{log_m['min']:.2f}",
        'Median': f"{log_m['median']:.2f}",
        'Mean': f"{log_m['mean']:.2f}",
        'p95': f"{log_m['p95']:.2f}",
        'Max': f"{log_m['max']:.2f}",
        'Std': f"{log_m['std']:.2f}",
        'Skewness': f"{log_m['skewness']:.2f}"
    }
])
df_trans_comp


## 5. Quantile Binning Collapse & Candidate Scoring Schemes

Because 97.00% of customers have $F=1$, standard quantile binning (`pd.qcut(F, q=5)`) fails with identical bin edges. When dropping duplicate edges, Frequency collapses into a single invariant bin (score 1 for 100% of customers). We evaluate alternative scoring strategies.

In [ ]:
scoring_data = audit_data['rfm_scoring_evaluation']
cand_eval = scoring_data['candidate_scoring_evaluations']

df_cand_scoring = pd.DataFrame([
    {
        'Candidate Scheme': 'Scheme A: Standard Quintiles (Drop Ties)',
        'Description': cand_eval['candidate_scheme_a_standard_quintiles_dropped_ties']['description'],
        'Unique Cells': cand_eval['candidate_scheme_a_standard_quintiles_dropped_ties']['unique_cells_produced'],
        'Assessment': cand_eval['candidate_scheme_a_standard_quintiles_dropped_ties']['assessment']
    },
    {
        'Candidate Scheme': 'Scheme B: Hybrid Discrete F (5-3-5)',
        'Description': cand_eval['candidate_scheme_b_hybrid_discrete_f']['description'],
        'Unique Cells': cand_eval['candidate_scheme_b_hybrid_discrete_f']['unique_cells_produced'],
        'Assessment': cand_eval['candidate_scheme_b_hybrid_discrete_f']['assessment']
    },
    {
        'Candidate Scheme': 'Scheme C: Simplified Domain Tiers (3-2-3)',
        'Description': cand_eval['candidate_scheme_c_simplified_domain_tiers']['description'],
        'Unique Cells': cand_eval['candidate_scheme_c_simplified_domain_tiers']['unique_cells_produced'],
        'Assessment': cand_eval['candidate_scheme_c_simplified_domain_tiers']['assessment']
    }
])

print('Quantile Collision Diagnostic:')
print(f"Error without drop: {scoring_data['frequency_quantile_collision_diagnostic']['error_encountered_without_drop']}")
print(f"Explanation: {scoring_data['frequency_quantile_collision_diagnostic']['explanation']}")

df_cand_scoring


## 6. Explicit Rule-Based Behavioral Customer Segmentation

To overcome the limitations of distance-based clustering on invariant frequency data, we establish 9 mutually exclusive, reproducible behavioral cohorts partitioned by explicit percentile thresholds.

In [ ]:
b_seg = audit_data['behavioral_segmentation']
rules = b_seg['rule_definitions']
profiles = b_seg['segment_profiles']

seg_rows = []
for seg_name, p in profiles.items():
    seg_rows.append({
        'Segment Name': seg_name,
        'Customer Count': f"{p['customer_count']:,}",
        'Customer Share': f"{p['customer_share_pct']:.2f}%",
        'Median Recency (d)': f"{p['median_recency_days']:.1f}",
        'Mean Freq': f"{p['mean_frequency']:.2f}",
        'Median Spend (BRL)': f"{p['median_observed_monetary_value_brl']:.2f}",
        'Total Spend (BRL)': f"{p['total_observed_monetary_value_brl']:,.2f}",
        'Monetary Share': f"{p['monetary_value_share_pct']:.2f}%"
    })

df_segments = pd.DataFrame(seg_rows).sort_values(by='Monetary Share', ascending=False)
print('=== EXPLICIT RULE DEFINITIONS ===')
for name, rule in rules.items():
    print(f"- {name}: {rule}")
print(f"\nTotal Classified: {b_seg['segmentation_completeness']['total_customers_classified']:,} (100.0%)")
df_segments


## 7. Cutoff-Based RFM Feature Construction (120d & 180d)

To prepare for leakage-safe predictive modeling, RFM metrics are computed strictly prior to observation cutoffs ($T_{obs} = \text{max\_ts} - W$). Orders occurring after $T_{obs}$ are strictly excluded from feature computation.

In [ ]:
cutoff_data = audit_data['cutoff_based_rfm_evaluations']

c_rows = []
for w_key, c_info in cutoff_data.items():
    r_c = c_info['recency_distribution']
    f_c = c_info['frequency_distribution']
    m_c = c_info['monetary_distribution']
    fb_c = c_info['frequency_breakdown']
    
    c_rows.append({
        'Cutoff Horizon': f"{c_info['future_window_days']} Days (W={c_info['future_window_days']})",
        'Cutoff T_obs': c_info['observation_cutoff_T_obs'].split('T')[0],
        'Eligible Customers': f"{c_info['eligible_customers_count']:,}",
        'F=1 Share': f"{fb_c['F_eq_1_pct']:.2f}%",
        'Repeat Share': f"{fb_c['F_ge_2_pct']:.2f}%",
        'Median Recency (d)': f"{r_c['median']:.1f}",
        'Median Spend (BRL)': f"{m_c['median']:.2f}",
        'Max Spend (BRL)': f"{m_c['max']:,.2f}"
    })

df_cutoffs = pd.DataFrame(c_rows)
print(f"120d Leakage Confirmation: {cutoff_data['cutoff_120d']['leakage_prevention_confirmation']}")
print(f"180d Leakage Confirmation: {cutoff_data['cutoff_180d']['leakage_prevention_confirmation']}")

df_cutoffs


## 8. Methodological Assessment & Decision Summary

Synthesize empirical findings across feature construction, differentiation dimensions, scoring validity, and deferred ML evaluations.

In [ ]:
dec = audit_data['methodological_decisions']

dec_rows = [
    {'Scope': 'RFM Feature Construction', 'Status': dec['rfm_feature_construction']['status'], 'Empirical Rationale': dec['rfm_feature_construction']['reason']},
    {'Scope': 'Recency Differentiation', 'Status': dec['recency_differentiation']['status'], 'Empirical Rationale': dec['recency_differentiation']['reason']},
    {'Scope': 'Frequency Differentiation', 'Status': dec['frequency_differentiation']['status'], 'Empirical Rationale': dec['frequency_differentiation']['reason']},
    {'Scope': 'Monetary Differentiation', 'Status': dec['monetary_differentiation']['status'], 'Empirical Rationale': dec['monetary_differentiation']['reason']},
    {'Scope': 'Conventional RFM Scoring', 'Status': dec['conventional_rfm_scoring']['status'], 'Empirical Rationale': dec['conventional_rfm_scoring']['reason']},
    {'Scope': 'Rule-Based Behavioral Segmentation', 'Status': dec['rule_based_behavioral_segmentation']['status'], 'Empirical Rationale': dec['rule_based_behavioral_segmentation']['reason']},
    {'Scope': 'K-Means Clustering Evaluation', 'Status': dec['kmeans_clustering']['status'], 'Empirical Rationale': dec['kmeans_clustering']['reason']},
    {'Scope': 'Churn Modeling & Window Choice', 'Status': dec['churn_modeling_and_window_selection']['status'], 'Empirical Rationale': dec['churn_modeling_and_window_selection']['reason']}
]

df_decisions = pd.DataFrame(dec_rows)
df_decisions


## 9. Checkpoint 5 Summary & Next Checkpoint

### Key Empirical Findings
- **Operational Customer Base**: 93,358 delivered customers successfully aggregated with zero missing values across Recency, Frequency, and Monetary dimensions.
- **Frequency Concentration**: 97.00% of customers have $F=1$; only 3.00% are repeat buyers ($F \ge 2$). This produces extreme quantile collapse on standard quintile binning.
- **Monetary Value Spread**: Observed spend ranges from 9.59 to 13,664.08 BRL (median 107.78 BRL, mean 165.17 BRL, skew 9.21; $\log_{1p}$ skew 0.53).
- **Behavioral Segmentation**: Explicit rules partition 100% of customers into 9 actionable cohorts. For example, *Recent High-Value Buyers* (3.69% of customers) contribute **10.00%** of total monetary spend.
- **Cutoff-Based RFM**: Leakage-safe feature tables verified at $T_{obs}^{120d}$ (68,977 customers) and $T_{obs}^{180d}$ (55,907 customers).
- **Raw Data Immutability**: All 9 raw Olist CSV files retained identical SHA-256 hashes.

### Next Steps
- Present Checkpoint 5 findings for review.
- Proceed to **Checkpoint 6: Clustering & Segmentation Evaluation (K-Means vs. Behavioral Fallback)** upon authorization.
- K-Means clustering and churn modeling window selection remain explicitly deferred.